In [7]:
# This is the code to screen job posting data
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from datetime import datetime
import gc, json, csv, re, os, glob

In [ ]:
def read_dat_data(curFile):
    colNames = ['招聘ID','公司ID','公司名称','城市名称','公司所在区域','工作薪酬','教育要求','工作经历',
                '工作描述','职位名称','工作名称','招聘数量','发布日期','行业名称','数据来源']
    resCSV = pd.read_csv(curFile, header=None, index_col=None, names=colNames,encoding='utf-8',quoting=csv.QUOTE_NONE, sep="@!", error_bad_lines=False, engine='python')
    return resCSV

In [ ]:
# The first purge of data:
# 1. We use 工作名称 as the title to feed the ChatGPT. 
# 2. We drop all the publish data is missing.
# 3. When 工作名称 is missing, we replace it with 职位名称.
# 4. We drop title of ``part-time"。
# 5. We drop same '公司ID', '工作名称', '城市名称', '工作描述' within a month, because we treat this case as duplicates (same job posting been published multiple times)
# define the file location
dataNameTmp = "E:/Data/job_posting/Raw_data/job_posting_%s.dat"

naSum = 0
dupSum = 0

curFile = dataNameTmp%21
        
print(curFile," is Grouping Computing...")
datDf = read_dat_data(curFile)
#datDf.loc[datDf['职位名称'].str.contains('%',na=False),'职位名称'] = np.NaN
 

In [ ]:
datDf = datDf.replace(r'\N',np.NaN).dropna(subset=['发布日期'])
datDf['工作名称'] = datDf[['工作名称']].replace(r'\N',np.NaN)
datDf = datDf[datDf['数据来源'].isin(['智联招聘', '前程无忧', '拉勾网', 'BOSS直聘', '58同城', '猎聘网', '看准网', '百姓网', '拉勾网', '猎聘', '赶集网', '博才网', 'BOSS'])]
datDf


In [ ]:

datDf.loc[datDf['工作名称'].isna(),'工作名称'] = datDf.loc[datDf['工作名称'].isna(),'职位名称']
datDf = datDf[datDf['工作名称'] != "兼职"]
    
curNa = datDf.shape[0]
naSum = naSum + curNa
    
datDf['date'] = datDf['发布日期'].apply(lambda x: x[0:7])
datDf = datDf.drop_duplicates(subset=['公司ID', '工作名称', '城市名称', 'date'], keep='first').reset_index(drop=True)
    
curDup = datDf.shape[0]
dupSum = dupSum + curDup
    
print(curFile,"删除空值剩余: %s"%curNa, "去重复值剩余：%s"%curDup)

In [ ]:
# Determine the data source, we limit to Top 10 job posting websites to avoid fuzzywuzzy in the data source.
dataNameTmp = "E:/Data/job_posting/Raw_data/job_posting_%s.dat"

df_list = []

for i in range(1,3): 
    curFile = dataNameTmp%i
        
    print(curFile," is Grouping Computing...")
    datDf = read_dat_data(curFile)


    # count number of occurrences of each value in column '数据来源', generate a new column to record the count
    datDf['count'] = datDf.groupby('数据来源')['数据来源'].transform('count')

    # drop duplicates based on column '数据来源', only keep the first occurrence
    datDf = datDf.drop_duplicates(subset=['数据来源'], keep='first').reset_index(drop=True)
    datDf = datDf[['数据来源', 'count']]

    df_list.append(datDf)
        
    #atDf.to_csv('F:/Data/job_posting/processed/temp/job_res_{}.csv'.format(i), sep='?', encoding = 'utf_8_sig', index=False)
final_df = pd.concat(df_list)

# group by '数据来源' and sum the count
final_df = final_df.groupby('数据来源').sum().reset_index()

# sort the dataframe based on column 'count'
final_df = final_df.sort_values(by=['count'], ascending=False)
final_df.head(20)

